In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pathlib import Path

In [ ]:
BASE_DIR_PATH = Path.cwd().parent
JDBC_DRIVER_PATH = f"{BASE_DIR_PATH}/drivers/postgresql-42.7.3.jar"
PG_HOST = ""
PG_PORT = 5432
PG_USER = ""
PG_PASSWORD = ""
PG_DATABASE = "vlr_events_metadata"
PG_TABLE = "agents"
jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
print(jdbc_url)

jdbc:postgresql://ep-misty-shadow-aipayev7-pooler.c-4.us-east-1.aws.neon.tech:5432/vlr_events_metadata


In [3]:
spark = (
    SparkSession.builder.master("local[*]")
    .config("spark.jars", JDBC_DRIVER_PATH)
    .appName("gold-pipeline")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

26/03/05 18:11:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [11]:
df = spark.read.parquet(f'{BASE_DIR_PATH}/data/silver')

In [12]:
total = df.count()
df = df.filter(
    (F.col("dq_null_core_fields") == False) & (F.col("dq_low_sample") == False)
)
clean = df.count()
print(f"Silver rows total   : {total:,}")
print(f"Silver rows after DQ filter : {clean:,}  ({total - clean:,} excluded)")

Silver rows total   : 699,383
Silver rows after DQ filter : 191,742  (507,641 excluded)


In [13]:
df.cache()

DataFrame[player_id: int, player: string, org: string, map: string, agent: string, role: string, snapshot_date: date, rounds_played: int, rating: double, average_combat_score: double, kill_death_ratio: double, kill_assists_survived_traded: double, average_damage_per_round: double, kills_per_round: double, assists_per_round: double, first_kills_per_round: double, first_deaths_per_round: double, headshot_percentage: double, clutch_success_percentage: double, clutches_won_played_ratio: double, max_kills_in_single_map: int, fk_fd_ratio: double, net_first_blood: int, damage_delta: double, kills: int, deaths: int, assists: int, first_kills: int, first_deaths: int, dq_low_sample: boolean, dq_clutch_no_attempts: boolean, dq_null_core_fields: boolean, dq_ratio_mismatch: boolean, event_id: int, region: string]

In [14]:
player_perf = df.groupBy("player_id", "player", "org", "event_id", "region").agg(
    # Volume
    F.sum("rounds_played").alias("total_rounds"),
    F.count("*").alias("map_agent_entries"),  # how many map+agent combos
    F.countDistinct("map").alias("maps_played"),
    F.countDistinct("agent").alias("agents_played"),
    # Core performance — weighted by presence (simple avg across entries)
    F.round(F.avg("rating"), 3).alias("avg_rating"),
    F.round(F.avg("average_combat_score"), 1).alias("avg_acs"),
    F.round(F.avg("kill_death_ratio"), 3).alias("avg_kd"),
    F.round(F.avg("kill_assists_survived_traded"), 3).alias("avg_kast"),
    F.round(F.avg("average_damage_per_round"), 1).alias("avg_adr"),
    F.round(F.avg("headshot_percentage"), 3).alias("avg_hs_pct"),
    # Derived metrics
    F.round(F.avg(F.when(F.col("fk_fd_ratio") != 99.0, F.col("fk_fd_ratio"))), 3).alias(
        "avg_fk_fd_ratio"
    ),  # excludes 99.0 sentinel
    F.round(F.avg("damage_delta"), 1).alias("avg_damage_delta"),
    # Raw totals — sum across all map+agent entries
    F.sum("kills").alias("total_kills"),
    F.sum("deaths").alias("total_deaths"),
    F.sum("assists").alias("total_assists"),
    F.sum("first_kills").alias("total_first_kills"),
    F.sum("first_deaths").alias("total_first_deaths"),
    # Clutch
    F.round(F.avg("clutch_success_percentage"), 3).alias("avg_clutch_pct"),
    # Peak performance
    F.max("average_combat_score").alias("peak_acs"),
    F.max("max_kills_in_single_map").alias("max_kills_single_map"),
    # Snapshot date — latest snapshot this player's data came from
    F.max("snapshot_date").alias("snapshot_date"),
)

In [15]:
agent_meta = df.groupBy("agent", "role", "event_id", "region").agg(
    # Pick rate signals
    F.countDistinct("player_id").alias("pick_count"),  # unique players
    F.countDistinct("map").alias("maps_appeared"),  # maps it was picked on
    F.sum("rounds_played").alias("total_rounds"),
    # Performance averages across all players who picked this agent
    F.round(F.avg("rating"), 3).alias("avg_rating"),
    F.round(F.avg("average_combat_score"), 1).alias("avg_acs"),
    F.round(F.avg("kill_death_ratio"), 3).alias("avg_kd"),
    F.round(F.avg("kill_assists_survived_traded"), 3).alias("avg_kast"),
    F.round(F.avg("average_damage_per_round"), 1).alias("avg_adr"),
    F.round(F.avg("headshot_percentage"), 3).alias("avg_hs_pct"),
    # Derived
    F.round(F.avg(F.when(F.col("fk_fd_ratio") != 99.0, F.col("fk_fd_ratio"))), 3).alias(
        "avg_fk_fd_ratio"
    ),
    F.round(F.avg("damage_delta"), 1).alias("avg_damage_delta"),
    F.max("snapshot_date").alias("snapshot_date"),
)

In [16]:
base = df.groupBy("map", "event_id", "region").agg(
    F.sum("rounds_played").alias("total_rounds"),
    F.countDistinct("player_id").alias("unique_players"),
    F.countDistinct("agent").alias("unique_agents"),
    F.round(F.avg("average_combat_score"), 1).alias("avg_acs"),
    F.round(F.avg("average_damage_per_round"), 1).alias("avg_adr"),
    F.round(F.avg("headshot_percentage"), 3).alias("avg_hs_pct"),
    F.round(F.avg("kill_death_ratio"), 3).alias("avg_kd"),
    F.round(F.avg("damage_delta"), 1).alias("avg_damage_delta"),
    F.max("snapshot_date").alias("snapshot_date"),
)

# Most picked agent per (map, event_id, region)
agent_window = Window.partitionBy("map", "event_id", "region").orderBy(
    F.col("agent_count").desc()
)
most_picked_agent = (
    df.groupBy("map", "event_id", "region", "agent")
    .agg(F.count("*").alias("agent_count"))
    .withColumn("rank", F.row_number().over(agent_window))
    .filter(F.col("rank") == 1)
    .select("map", "event_id", "region", F.col("agent").alias("most_picked_agent"))
)

# Most picked role per (map, event_id, region)
role_window = Window.partitionBy("map", "event_id", "region").orderBy(
    F.col("role_count").desc()
)
most_picked_role = (
    df.groupBy("map", "event_id", "region", "role")
    .agg(F.count("*").alias("role_count"))
    .withColumn("rank", F.row_number().over(role_window))
    .filter(F.col("rank") == 1)
    .select("map", "event_id", "region", F.col("role").alias("most_picked_role"))
)

# Join everything together
map_stats = base.join(
    most_picked_agent, on=["map", "event_id", "region"], how="left"
).join(most_picked_role, on=["map", "event_id", "region"], how="left")

In [17]:
def write_gold(df: any, gold_path: str, table_name: str):
    """
    Write a Gold table as Parquet partitioned by event_id.

    WHY partition by event_id only:
      Gold is aggregated — far fewer rows than Silver.
      One partition column is enough. BigQuery will use it for
      partition pruning when analysts filter by event.
    """
    output_path = f"{gold_path}/{table_name}"
    row_count = df.count()

    (
        df.write.format("parquet")
        .mode("overwrite")
        .partitionBy("event_id")
        .save(output_path)
    )

    print(f"  {table_name:<30} {row_count:>8,} rows → {output_path}")

In [ ]:
write_gold(player_perf, f"{BASE_DIR_PATH}/data/gold", "gold_player_performance")
write_gold(agent_meta, f"{BASE_DIR_PATH}/data/gold", "gold_agent_meta")
write_gold(map_stats, f"{BASE_DIR_PATH}/data/gold", "gold_map_stats")

  gold_player_performance          61,664 rows → data/gold/gold_player_performance


  gold_agent_meta                   4,908 rows → data/gold/gold_agent_meta


  gold_map_stats                    1,952 rows → data/gold/gold_map_stats
